# Checkpoint export


In [ ]:
import json
import struct
from pathlib import Path

import numpy as np
import torch

import import_ipynb
from model_architecture import ASRModel, CONFIGS
from tokenizer_training import MultitaskTokenizer, BPETokenizer, base_alphabet, merges

In [ ]:
RUN = "runs/tiny-multitask-v1"
EXPORT = Path("export/tiny")
EXPORT.mkdir(parents=True, exist_ok=True)

bpe = BPETokenizer(base_alphabet, merges)
tokenizer = MultitaskTokenizer(bpe)
dims = CONFIGS["tiny"]
dims.n_vocab = tokenizer.n_vocab

checkpoints = sorted(Path(RUN).glob("step-*.pt"))
state = torch.load(checkpoints[-1], map_location="cpu", weights_only=False)
model = ASRModel(dims)
model.load_state_dict(state["model"])
model.eval()
print(checkpoints[-1], state["step"])

In [ ]:
DTYPES = {torch.float32: "F32", torch.float16: "F16", torch.bfloat16: "BF16"}

def save_safetensors(tensors, path):
    header = {}
    offset = 0
    blobs = []
    for name in sorted(tensors):
        t = tensors[name].contiguous()
        if t.dtype == torch.bfloat16:
            data = t.view(torch.uint16).numpy().tobytes()
        else:
            data = t.numpy().tobytes()
        header[name] = {
            "dtype": DTYPES[t.dtype],
            "shape": list(t.shape),
            "data_offsets": [offset, offset + len(data)],
        }
        offset += len(data)
        blobs.append(data)
    header_bytes = json.dumps(header, separators=(",", ":")).encode()
    pad = (8 - len(header_bytes) % 8) % 8
    header_bytes += b" " * pad
    with open(path, "wb") as f:
        f.write(struct.pack("<Q", len(header_bytes)))
        f.write(header_bytes)
        for b in blobs:
            f.write(b)

def load_safetensors(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(n))
        body = f.read()
    out = {}
    revd = {v: k for k, v in DTYPES.items()}
    for name, meta in header.items():
        if name == "__metadata__":
            continue
        a, b = meta["data_offsets"]
        dtype = revd[meta["dtype"]]
        if dtype == torch.bfloat16:
            t = torch.frombuffer(bytearray(body[a:b]), dtype=torch.uint16).view(torch.bfloat16)
        else:
            np_dtype = {torch.float32: np.float32, torch.float16: np.float16}[dtype]
            t = torch.from_numpy(np.frombuffer(body[a:b], dtype=np_dtype).copy())
        out[name] = t.view(meta["shape"])
    return out

In [ ]:
RENAMES = [
    ("encoder.", "model.encoder."),
    ("decoder.", "model.decoder."),
    (".attn.query.", ".self_attn.q_proj."),
    (".attn.key.", ".self_attn.k_proj."),
    (".attn.value.", ".self_attn.v_proj."),
    (".attn.out.", ".self_attn.out_proj."),
    (".cross_attn.query.", ".encoder_attn.q_proj."),
    (".cross_attn.key.", ".encoder_attn.k_proj."),
    (".cross_attn.value.", ".encoder_attn.v_proj."),
    (".cross_attn.out.", ".encoder_attn.out_proj."),
    (".attn_ln.", ".self_attn_layer_norm."),
    (".cross_attn_ln.", ".encoder_attn_layer_norm."),
    (".mlp.0.", ".fc1."),
    (".mlp.2.", ".fc2."),
    (".mlp_ln.", ".final_layer_norm."),
    ("blocks.", "layers."),
    ("ln_post.", "layer_norm."),
    ("decoder.ln.", "decoder.layer_norm."),
    ("token_embedding.", "embed_tokens."),
    ("positional_embedding", "embed_positions.weight"),
]

def export_name(name):
    for old, new in RENAMES:
        name = name.replace(old, new)
    return name

tensors = {}
for name, p in model.state_dict().items():
    tensors[export_name(name)] = p.to(torch.float32)
save_safetensors(tensors, EXPORT / "model.safetensors")
print(len(tensors), (EXPORT / "model.safetensors").stat().st_size)

In [ ]:
config = {
    "architectures": ["KikuForConditionalGeneration"],
    "model_type": "kiku",
    "num_mel_bins": dims.n_mels,
    "max_source_positions": dims.n_audio_ctx,
    "d_model": dims.n_audio_state,
    "encoder_attention_heads": dims.n_audio_head,
    "encoder_layers": dims.n_audio_layer,
    "decoder_attention_heads": dims.n_text_head,
    "decoder_layers": dims.n_text_layer,
    "max_target_positions": dims.n_text_ctx,
    "vocab_size": dims.n_vocab,
    "decoder_start_token_id": tokenizer.sot,
    "eos_token_id": tokenizer.eot,
}
with open(EXPORT / "config.json", "w") as f:
    json.dump(config, f, indent=2)

In [ ]:
import shutil
shutil.copy("assets/tokenizer.json", EXPORT / "tokenizer.json")
print(sorted(p.name for p in EXPORT.iterdir()))

In [ ]:
reloaded = load_safetensors(EXPORT / "model.safetensors")
mismatches = []
for name, p in model.state_dict().items():
    q = reloaded[export_name(name)]
    if not torch.allclose(p.to(torch.float32), q, atol=0):
        mismatches.append(name)
print(len(reloaded), mismatches)

In [ ]:
from decoding import transcribe, DecodeOptions
import soundfile as sf

wave, _ = sf.read("data/samples/meeting-excerpt.wav", dtype="float32")
if wave.ndim > 1:
    wave = wave.mean(axis=1)
before = [s.text for s in transcribe(model, tokenizer, wave)]

model2 = ASRModel(dims)
inverse = {export_name(n): n for n in model.state_dict()}
model2.load_state_dict({inverse[n]: t for n, t in reloaded.items()})
model2.eval()
after = [s.text for s in transcribe(model2, tokenizer, wave)]
print(before == after)

In [ ]:
manifest = {
    "size": "tiny",
    "step": int(state["step"]),
    "n_vocab": dims.n_vocab,
    "files": {
        p.name: p.stat().st_size for p in EXPORT.iterdir()
    },
}
with open(EXPORT / "export-manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))